# Featuresmith Tutorial: 08 — Recommendations and Plans

Learn how Featuresmith's centralized Recommendation Engine transforms review findings into actionable recommendations, and how to compile accepted recommendations into a deterministic, inspectable Plan.

---


## 1. From Findings to Recommendations to Plan

Featuresmith's v0.4.0 introduces a centralized Recommendation Engine and the Plan primitive, completing the loop from detection to action:

```
Review (findings) → Recommendation Engine → Recommendations → Plan (accepted items)
```

- **Review**: `fs.review()` runs reviewers and produces `ReviewResult` with findings grouped into sections.
- **Recommendation Engine**: `RecommendationEngine.generate(sections)` merges findings by column and rule pattern, producing ranked `Recommendation` objects with traceability to originating findings and reviewers.
- **Recommendations**: Each `Recommendation` has a stable ID (`rec.{rule_prefix}.{column}`), title, rationale, confidence (0–1), severity, affected columns, suggested action, and full traceability (`originating_findings`, `originating_reviewers`).
- **Plan**: `fs.plan(result, accept=[rec_ids])` compiles a deterministic `Plan` from accepted recommendation IDs. Each `PlanItem` inherits all traceability and has a deterministic ID `plan.{rec_id}.{idx}`.

This creates a complete chain: **PlanItem → Recommendation → Finding → Reviewer**.


## 2. Prerequisites

This notebook uses the Titanic dataset. From the repository root, run:

```bash
python examples/prepare_datasets.py
```


In [1]:
import os

import featuresmith as fs

data_path = os.path.join("..", "data", "processed", "titanic.csv")
dataset = fs.load(data_path)
print(f"Dataset loaded: {dataset.row_count} rows, {dataset.column_count} columns")

Dataset loaded: 891 rows, 12 columns


## 3. Run Review and Inspect Recommendations

Run `fs.review()` with a target column to get leakage detection and full recommendations.

In [2]:
review_result = fs.review(dataset, target_column="survived")

print(f"Total sections: {len(review_result.sections)}")
print(f"Total recommendations: {len(review_result.recommendations)}")

print("\nRecommendations:")
for rec in review_result.recommendations:
    print(f"  [{rec.severity.upper()}] {rec.title} — {rec.suggested_action}")

Total sections: 9
Total recommendations: 8

Recommendations:
  [CRITICAL] Fix missing value threshold in column 'cabin' — Address the flagged issue: High missing values in column 'cabin'.
  [WARNING] Fix basic statistics in column 'sibsp' — Address the flagged issue: High skewness in column 'sibsp'.
  [WARNING] Fix basic statistics in column 'parch' — Address the flagged issue: High skewness in column 'parch'.
  [WARNING] Fix basic statistics in column 'fare' — Address the flagged issue: High skewness in column 'fare'.
  [INFO] Fix data types in column 'passengerid' — Address the flagged issue: Identifier-like column 'passengerid'.
  [INFO] Fix data types in column 'name' — Address the flagged issue: Text column 'name'.
  [INFO] Fix data types in column 'ticket' — Address the flagged issue: Text column 'ticket'.
  [INFO] Fix data types in column 'cabin' — Address the flagged issue: Text column 'cabin'.


## 4. Recommendation Details and Traceability

Each recommendation carries full traceability to its originating findings and reviewers.

In [3]:
for rec in review_result.recommendations:
    print(f"Recommendation ID: {rec.id}")
    print(f"  Title: {rec.title}")
    print(f"  Severity: {rec.severity}")
    print(f"  Confidence: {rec.confidence}")
    print(f"  Affected columns: {rec.affected_columns}")
    print(f"  Suggested action: {rec.suggested_action}")
    print(f"  Originating findings: {len(rec.originating_findings)}")
    for f in rec.originating_findings:
        print(
            f"    - Rule: {f.rule_id} | Column: {f.column_name} | Severity: {f.severity}"
        )
    print(f"  Originating reviewers: {', '.join(rec.originating_reviewers)}")
    print()

Recommendation ID: rec.quality.cabin
  Title: Fix missing value threshold in column 'cabin'
  Severity: critical
  Confidence: 1.0
  Affected columns: ('cabin',)
  Suggested action: Address the flagged issue: High missing values in column 'cabin'.
  Originating findings: 1
    - Rule: quality.missing_value_threshold | Column: cabin | Severity: critical
  Originating reviewers: review.quality.missingness

Recommendation ID: rec.review.sibsp
  Title: Fix basic statistics in column 'sibsp'
  Severity: warning
  Confidence: 1.0
  Affected columns: ('sibsp',)
  Suggested action: Address the flagged issue: High skewness in column 'sibsp'.
  Originating findings: 2
    - Rule: review.quality.basic_statistics | Column: sibsp | Severity: warning
    - Rule: review.quality.basic_statistics | Column: sibsp | Severity: info
  Originating reviewers: review.quality.basic_statistics

Recommendation ID: rec.review.parch
  Title: Fix basic statistics in column 'parch'
  Severity: warning
  Confidence: 

## 5. Creating a Plan from Accepted Recommendations

Use `fs.plan()` to compile a deterministic Plan from accepted recommendation IDs. The Plan is the central domain primitive for the Dataset Contract lifecycle.

In [4]:
# Accept the first two recommendations dynamically
accepted_ids = [
    review_result.recommendations[0].id,
    review_result.recommendations[1].id,
]

plan = fs.plan(review_result, accept=accepted_ids)

# Render the plan
plan_text = fs.render(plan, "console")
print(plan_text)

Featuresmith Plan
Plan Schema Version: 0.1.0
Accepted Recommendations: 2
Plan Items: 2

1. [CRITICAL] Fix missing value threshold in column 'cabin'
   ID: plan.rec.quality.cabin.0
   From Recommendation: rec.quality.cabin
   Confidence: 1.00
   Severity: critical
   Affected Columns: cabin
   Action: Address the flagged issue: High missing values in column 'cabin'.
   Rationale: Column 'cabin' has 77.10% missing values, exceeding the threshold of 20.00%.
   Originating Findings: 1
   Originating Reviewers: review.quality.missingness

2. [WARNING] Fix basic statistics in column 'sibsp'
   ID: plan.rec.review.sibsp.1
   From Recommendation: rec.review.sibsp
   Confidence: 1.00
   Severity: warning
   Affected Columns: sibsp
   Action: Address the flagged issue: High skewness in column 'sibsp'.
   Rationale: Multiple related issues: Column 'sibsp' has skewness 3.69, exceeding the threshold of 2.00.; Column 'sibsp' has kurtosis 17.77, exceeding the threshold of 10.00.
   Originating Findin

## 6. Plan Serialization and SDK/CLI Parity

Plans are fully serializable. The SDK `fs.plan()` and CLI `featuresmith plan` produce identical canonical Plans.

In [5]:
import json

# SDK Plan serialization
plan_json = json.dumps(plan.to_dict(), indent=2)
print("Plan JSON (first 500 chars):")
print(plan_json[:500] + "...")

# Verify Plan structure
plan_data = json.loads(plan_json)
assert plan_data["plan_schema_version"] == "0.1.0"
assert len(plan_data["items"]) == 2
assert plan_data["accepted_recommendation_ids"] == accepted_ids
print("\nPlan structure validation passed.")

Plan JSON (first 500 chars):
{
  "plan_schema_version": "0.1.0",
  "items": [
    {
      "id": "plan.rec.quality.cabin.0",
      "recommendation_id": "rec.quality.cabin",
      "title": "Fix missing value threshold in column 'cabin'",
      "rationale": "Column 'cabin' has 77.10% missing values, exceeding the threshold of 20.00%.",
      "confidence": 1.0,
      "severity": "critical",
      "affected_columns": [
        "cabin"
      ],
      "suggested_action": "Address the flagged issue: High missing values in column 'c...

Plan structure validation passed.


## 7. CLI Usage

The same Plan can be generated from the CLI:

```bash
# First, run review to see available recommendation IDs
featuresmith review titanic.csv --target survived --format json --output review.json

# Then create a plan from accepted recommendation IDs
featuresmith plan titanic.csv --target survived --accept rec.quality.cabin,rec.review.name --format json --output plan.json
```

The CLI supports `--format table|json`, `--fail-on critical|warning|info`, `--output`, `--quiet`, and `--previous` for diff-aware plans.

## 8. Key Takeaways

- **Recommendation Engine**: Centralized, merges findings from all sections into ranked recommendations with consistent confidence semantics and full traceability.
- **Recommendation**: Stable ID, title, rationale, confidence, severity, affected columns, suggested action, `accepted` flag, `originating_findings`, `originating_reviewers`.
- **Plan**: Deterministic, inspectable, serializable (`PlanItem` → `Recommendation` → `Finding` → `Reviewer`).
- **`fs.plan()`**: Compiles Plan from accepted recommendation IDs; validates IDs against available recommendations.
- **Plan Rendering**: `fs.render(plan, "console")` or `featuresmith plan --format table|json`.
- **Traceability**: Every PlanItem traces back to originating findings and reviewers — complete evidence chain.

**Next Steps**: In `05_dataset_diff.ipynb`, we explore diff-aware reviews. In future tutorials, we'll cover the Export/Apply step and Dataset Contracts (`featuresmith.lock`).